## Why bare pods aren't enough

Notebook 02 created pods directly. That works for learning but falls over in production for two reasons.

- **Bare pods don't survive node failure.** A `Pod` object in `etcd` references a specific node. If that node dies, its kubelet stops reporting in, the pod's phase drifts to `Unknown`, and *nothing* moves it elsewhere. The pod is simply gone.
- **Bare pods don't roll forward.** To go from `nginx:1.27` to `1.28` you'd delete the old pod and create a new one — a window with zero pods running. For a long-running service that's an outage.

What you want is a **higher-level object** that says: "keep three copies of *this template* alive anywhere in the cluster, and when I change the template, replace the pods gracefully." That's the **Deployment**.

### Controllers — the missing word

A **controller** is a program (usually in `kube-controller-manager`) that watches one kind of object and works to make reality match its `spec`. You've already met one: the kubelet is a controller for pods on its node. Others:

- **ReplicaSet controller** — the right number of matching pods exist.
- **Deployment controller** — manages ReplicaSets, creating new ones during rollouts.
- **DaemonSet** — one pod per node. **StatefulSet** — stable identity + storage. **Job** — run to completion. **Node controller** — marks nodes unreachable when heartbeats stop.

Every one is the same loop: read desired state from `etcd`, observe reality, act to close the gap, write `status` back. On our map, this is why the bare **Pods** box can't stand alone — it needs a controller in the Workloads band above it to give it resurrection and rollout. Almost nobody runs bare pods in production for exactly these two reasons.